# Tema 11 · Bloque 2 — El Hamiltoniano, Qiskit Nature y Clásico vs. Cuántico
### Solución completa de los 4 retos (100% local / offline, sin token de IBM Quantum)

**Cómo usar este notebook:** ejecuta las celdas de arriba hacia abajo. Cada reto tiene:
1. Una sección **"Entiende esto primero"** con los conceptos en lenguaje llano.
2. El **código** que produce el resultado.
3. La **respuesta escrita** a la pregunta del reto (lista para entregar o exponer).

---

## Mapa mental del tema en 6 líneas

| Concepto | Qué es, en cristiano |
|---|---|
| **Hamiltoniano (H)** | La "hoja de cálculo" de toda la energía de una molécula: cinética + potencial. Es un **operador** = una **matriz**. |
| **Estado fundamental** | La configuración de **mínima energía**. La naturaleza siempre cae ahí. |
| **Eigenvalue (valor propio)** | Los niveles de energía permitidos de esa matriz. El **más pequeño** es el estado fundamental. |
| **Operadores de Pauli (I, X, Y, Z)** | Las 4 matrices de 2×2 que son el "alfabeto" del hardware cuántico. Todo H se escribe como suma de ellas. |
| **Jordan-Wigner** | El traductor: convierte electrones (fermiones) en qubits. |
| **NISQ / ZNE** | Hoy el hardware tiene ruido; ZNE es un truco de software para corregirlo después de medir. |

> **La frase que resume el tema entero:** *calcular química cuántica = escribir la energía de la molécula como una matriz, y buscar su valor propio más pequeño.*


## 0. Preparación del entorno

Solo se necesitan `qiskit` (la parte `quantum_info`, que es puro cálculo local), `numpy` y `matplotlib`.
**No hace falta cuenta ni token de IBM**: nada de esto se ejecuta en la nube.

> 📌 **En Google Colab** `qiskit` no viene preinstalado. La celda de abajo lo detecta y lo instala sola
> (tarda ~30 s la primera vez). Si prefieres hacerlo a mano: `!pip install -q qiskit`.
> Si la instalación termina y sigue fallando el import, reinicia el entorno de ejecución
> (*Entorno de ejecución → Reiniciar sesión*) y vuelve a ejecutar desde el principio.


In [ ]:
# Instala qiskit automáticamente si falta (Google Colab NO lo trae preinstalado).
# Tarda ~30 s la primera vez. numpy y matplotlib ya vienen incluidos en Colab.
try:
    import qiskit
except ModuleNotFoundError:
    print("qiskit no encontrado -> instalando...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qiskit"], check=True)
    import qiskit

import numpy as np
import matplotlib.pyplot as plt
from qiskit.quantum_info import SparsePauliOp, Operator

print("Entorno listo · qiskit", qiskit.__version__)
print("Todo corre en local, sin credenciales ni token de IBM.")


---
# RETO 1 — Definición conceptual y mapeo local del Hamiltoniano (H₂)

## 🧠 Entiende esto primero (aquí están TODOS los conceptos del reto)

### 1) ¿Qué es un "operador" y por qué es una matriz?
En cuántica, una magnitud física (energía, posición, espín) no es un número: es un **operador**, y sobre un sistema
finito de qubits ese operador se escribe como una **matriz cuadrada**. El Hamiltoniano `H` es el operador de la
**energía total**. Si el sistema tiene `n` qubits, `H` es una matriz de `2ⁿ × 2ⁿ`. Para H₂ en base mínima (STO-3G)
usamos **2 qubits**, así que `H` es una matriz de **4×4**.

### 2) Las cuatro matrices de Pauli = el alfabeto
Cualquier matriz hermítica (y `H` lo es, porque la energía es real) se puede escribir como **suma de productos
tensoriales de Pauli**. Solo existen cuatro letras:

$$ I=\begin{pmatrix}1&0\\0&1\end{pmatrix},\;
X=\begin{pmatrix}0&1\\1&0\end{pmatrix},\;
Y=\begin{pmatrix}0&-i\\i&0\end{pmatrix},\;
Z=\begin{pmatrix}1&0\\0&-1\end{pmatrix} $$

Intuición física de cada letra:
- **I** → "no le hagas nada a este qubit" (término constante / de referencia).
- **Z** → *mide ocupación*: devuelve +1 si el orbital está vacío y −1 si está ocupado (o al revés, según convenio).
  Por eso **Z describe energías diagonales**: "cuánto cuesta que este orbital esté lleno".
- **X, Y** → *mueven* electrones: intercambian |0⟩ ↔ |1⟩. Describen **saltos / correlación**: un electrón que
  salta de un orbital a otro. Son los términos **fuera de la diagonal**.

### 3) ¿Qué significa una cadena como `IZ`, `ZI`, `ZZ`, `XX`?
Cada letra es **un qubit** (= un espín-orbital). La cadena es un **producto tensorial**:

- `II` = I⊗I → energía de referencia (offset constante, incluye la repulsión nuclear).
- `IZ` = "Z sobre el qubit 0" → energía asociada a la ocupación **del orbital 0**.
- `ZI` = "Z sobre el qubit 1" → energía asociada a la ocupación **del orbital 1**.
- `ZZ` → energía de **interacción** entre ambos orbitales (repulsión coulombiana electrón-electrón: cuesta
  distinto si ambos están ocupados a la vez).
- `XX` → **término de correlación / salto doble**: los dos electrones cambian de orbital simultáneamente.
  Es el término que la química clásica de campo medio (Hartree-Fock) **no captura bien**.

> ⚠️ Convenio de Qiskit: en la cadena, el qubit **0 es el de más a la derecha** (little-endian). Por eso `IZ`
> significa Z en el qubit 0, y `ZI` significa Z en el qubit 1.

### 4) ¿Qué son los coeficientes (−1.05, 0.39, …)?
Son los **pesos energéticos**: cuánta energía aporta cada interacción. Salen de las *integrales moleculares*
(integrales de uno y dos cuerpos) calculadas clásicamente para una distancia de enlace concreta (0.735 Å en H₂).
Se expresan en **Hartree** (1 Ha ≈ 27.2 eV). Cambiar la distancia entre núcleos **cambia estos coeficientes**.

### 5) ¿Qué es `SparsePauliOp` y por qué "sparse"?
Es la clase de Qiskit que guarda un operador como **lista de (cadena de Pauli, coeficiente)** en vez de guardar
la matriz completa. Para 2 qubits da igual (4×4 = 16 números), pero para 50 qubits la matriz densa tendría
2⁵⁰×2⁵⁰ entradas — imposible. Guardando solo los términos que existen (**representación dispersa**), el coste
crece con el **número de interacciones reales**, no con 4ⁿ. Ese es el truco que hace viable la química cuántica.

### 6) ¿Y el `JordanWignerMapper`?
Los electrones son **fermiones**: al intercambiar dos, la función de onda cambia de signo (principio de Pauli).
Los qubits **no** tienen esa propiedad. **Jordan-Wigner** es la receta que traduce operadores fermiónicos
(crear/aniquilar un electrón en un orbital) a cadenas de Pauli, añadiendo colas de `Z` que llevan la cuenta del
signo. Resultado: `1 espín-orbital → 1 qubit`. El código de este reto usa directamente el **resultado ya mapeado**
(los términos y coeficientes), para no depender de PySCF ni de red.


## 1.1 Código base del reto

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Representación simbólica local del hamiltoniano simplificado de H2
# Términos representativos en base de Pauli (Z, X) con coeficientes de energía
terms  = ["II", "IZ", "ZI", "ZZ", "XX"]
coeffs = [-1.05, 0.39, 0.39, -0.01, 0.18]

hamiltonian_simbolico = SparsePauliOp.from_list(list(zip(terms, coeffs)))

print("Operador Hamiltoniano mapeado:")
print(hamiltonian_simbolico)


## 1.2 Leamos el objeto por dentro (qué guarda y qué NO guarda)

In [ ]:
significado = {
    "II": "Energía de referencia (offset + repulsión nuclear). No depende de la ocupación.",
    "IZ": "Energía del espín-orbital 0 (qubit 0): ¿cuánto cuesta ocuparlo?",
    "ZI": "Energía del espín-orbital 1 (qubit 1): ¿cuánto cuesta ocuparlo?",
    "ZZ": "Interacción electrón-electrón: energía extra si ambos orbitales están ocupados.",
    "XX": "Correlación / doble excitación: los dos electrones saltan de orbital a la vez.",
}

print(f"{'Pauli':<6}{'Coef (Ha)':>12}   Significado físico")
print("-" * 100)
for pauli, c in zip(hamiltonian_simbolico.paulis, hamiltonian_simbolico.coeffs):
    print(f"{str(pauli):<6}{c.real:>12.4f}   {significado[str(pauli)]}")

print("\nNº de qubits           :", hamiltonian_simbolico.num_qubits)
print("Nº de términos guardados:", len(hamiltonian_simbolico))
print("Tamaño de la matriz densa equivalente:",
      f"{2**hamiltonian_simbolico.num_qubits} x {2**hamiltonian_simbolico.num_qubits}")
print("Términos posibles con 2 qubits (4^n) :", 4**hamiltonian_simbolico.num_qubits,
      "-> guardamos solo", len(hamiltonian_simbolico), "(eso es 'sparse')")


## 1.3 Comprobación: la suma de Paulis ES la matriz de energía

Reconstruimos la matriz 4×4 a mano (con productos tensoriales) y verificamos que coincide con la que da Qiskit.
Esto demuestra que **"suma ponderada de Paulis" y "matriz de energía" son lo mismo escrito de dos formas**.


In [ ]:
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
P = {"I": I, "X": X, "Z": Z}

def cadena_a_matriz(cadena):
    # Convierte 'IZ' en el producto tensorial correcto (convenio little-endian de Qiskit).
    M = np.array([[1]], dtype=complex)
    for letra in cadena:          # se recorre de izquierda a derecha = qubit n-1 ... qubit 0
        M = np.kron(M, P[letra])
    return M

H_manual = sum(c * cadena_a_matriz(t) for t, c in zip(terms, coeffs))
H_qiskit = hamiltonian_simbolico.to_matrix()

np.set_printoptions(precision=3, suppress=True)
print("Matriz del Hamiltoniano (4x4), parte real:\n")
print(H_manual.real)
print("\n¿Coincide con la matriz de Qiskit?", np.allclose(H_manual, H_qiskit))
print("¿Es hermítica (energía real)?     ", np.allclose(H_manual, H_manual.conj().T))

print("\nLectura de la matriz:")
print("  - La DIAGONAL viene de los términos con I y Z  -> energías de cada configuración electrónica.")
print("  - Los elementos FUERA de la diagonal (0.18) vienen de XX -> acoplan |00> con |11>: correlación.")


## ✅ Respuesta al Reto 1

**Pregunta:** *¿por qué el hamiltoniano molecular se descompone en una suma ponderada de operadores de Pauli
tensoriales (como IZ, ZI, XX) y qué representa cada término físico?*

**Respuesta:**

Se descompone así por **tres razones encadenadas**:

1. **Razón matemática.** El Hamiltoniano es un operador hermítico que actúa sobre un espacio de Hilbert de
   dimensión 2ⁿ. Los productos tensoriales de las matrices de Pauli `{I, X, Y, Z}⊗ⁿ` forman una **base completa
   y ortogonal** de ese espacio de operadores. Por tanto, *cualquier* Hamiltoniano puede escribirse de forma
   única como `H = Σᵢ cᵢ Pᵢ`, donde `Pᵢ` es una cadena de Paulis y `cᵢ` un coeficiente real.

2. **Razón de hardware.** Una computadora cuántica **no sabe leer una matriz**; sabe aplicar compuertas y medir
   en la base de Pauli. Escribir `H` como suma de Paulis convierte el problema de química en algo directamente
   **medible**: el valor esperado de la energía es `⟨H⟩ = Σᵢ cᵢ ⟨Pᵢ⟩`, y cada `⟨Pᵢ⟩` se obtiene con una rotación
   de base y una medición. Es literalmente traducir química al lenguaje nativo de la máquina.

3. **Razón de eficiencia.** El mapeo Jordan-Wigner traduce los operadores fermiónicos de creación/aniquilación en
   cadenas de Pauli; el número de términos crece **polinómicamente** (≈ N⁴ con los orbitales), no
   exponencialmente. Guardarlos en un `SparsePauliOp` (formato disperso) significa almacenar solo las
   interacciones que realmente existen, en vez de una matriz densa de 2ⁿ×2ⁿ.

**Qué representa físicamente cada término:**

| Término | Tipo | Significado físico |
|---|---|---|
| `II` | Constante | Energía de referencia del sistema: offset de las integrales + **repulsión nuclear** (depende solo de la distancia entre núcleos, no de los electrones). |
| `IZ` | Diagonal, 1 cuerpo | **Energía del espín-orbital 0**: energía cinética del electrón + atracción núcleo-electrón asociada a ocupar ese orbital. `Z` "pregunta" si está ocupado (±1). |
| `ZI` | Diagonal, 1 cuerpo | Lo mismo para el **espín-orbital 1**. |
| `ZZ` | Diagonal, 2 cuerpos | **Repulsión electrón-electrón**: energía adicional cuando ambos orbitales están ocupados simultáneamente. |
| `XX` | Fuera de diagonal, 2 cuerpos | **Correlación electrónica / doble excitación**: amplitud de que ambos electrones salten de orbital a la vez. Acopla `|00⟩` con `|11⟩` y es lo que hace que la energía real sea menor que la de Hartree-Fock. |

**En una frase:** los términos con `Z` son *"cuánta energía cuesta que los orbitales estén ocupados"* (estática,
diagonal), y los términos con `X`/`Y` son *"cómo se mueven y correlacionan los electrones"* (dinámica, fuera de la
diagonal). La suma ponderada de ambos es la energía total de la molécula.


---
# RETO 2 — El Hamiltoniano como partitura de la energía total

## 🧠 Entiende esto primero
- **Eigenvalue (valor propio):** número `E` tal que `H|ψ⟩ = E|ψ⟩`. Físicamente: un **nivel de energía permitido**.
- Una matriz 4×4 tiene **4 eigenvalues** → el espectro de energías del sistema.
- **Estado fundamental (ground state):** el eigenvalue **mínimo**. La molécula estable vive ahí.
- **Hartree (Ha):** unidad de energía atómica. 1 Ha ≈ 27.211 eV ≈ 627.5 kcal/mol.
- **Precisión química:** 1 kcal/mol ≈ **0.0016 Ha**. Ese es el error máximo tolerable para que un cálculo sirva
  para diseñar un fármaco o un catalizador.


In [ ]:
# Diagonalización exacta del Hamiltoniano (para 2 qubits es trivial en clásico)
H = hamiltonian_simbolico.to_matrix()
eigenvalues, eigenvectors = np.linalg.eigh(H)   # eigh = matrices hermíticas, devuelve ordenado

print("Espectro de energía (eigenvalues, en Hartree):")
print(np.round(eigenvalues, 4))

E0 = eigenvalues[0]
print(f"\n>>> Estado fundamental (ground state): {E0:.4f} Hartree")
print(f"    Gap al primer excitado             : {eigenvalues[1]-E0:.4f} Ha  "
      f"({(eigenvalues[1]-E0)*27.2114:.2f} eV)")

psi0 = eigenvectors[:, 0]
base = ["|00>", "|01>", "|10>", "|11>"]
print("\nComposición del estado fundamental (probabilidades):")
for b, amp in zip(base, psi0):
    print(f"   {b}: amplitud {amp.real:+.4f}   probabilidad {abs(amp)**2:.4f}")
print("\nOJO: el estado fundamental NO es un estado puro de la base: es una SUPERPOSICIÓN.")
print("Domina |11> (la configuración de Hartree-Fock), pero hay una contribución de |00>.")
print("Esa mezcla es exactamente la CORRELACIÓN ELECTRÓNICA que introduce el término XX:")
print("es la parte de la energía que Hartree-Fock NO puede capturar.")


## 2.1 Cómo la distancia de enlace cambia la energía

Los coeficientes del Hamiltoniano **son función de la distancia interatómica R**. Al alejar los núcleos:
- la **repulsión nuclear** (`II`) cae como `1/R`,
- la **atracción núcleo-electrón** se debilita y el **solapamiento** de orbitales (términos `XX`) se desvanece.

El mínimo de la curva `E(R)` es la **geometría de equilibrio** (para H₂: ≈ 0.735 Å). Abajo usamos un modelo
analítico sencillo (tipo Morse) para *ver* el efecto sin necesidad de PySCF ni de red.


In [ ]:
def coeficientes_H2(R):
    # MODELO DIDÁCTICO (no son las integrales exactas de PySCF, pero reproducen
    # las tendencias físicas correctas y coinciden con el Hamiltoniano del Reto 1 en R = 0.735 A):
    #   - c_II: competencia entre repulsión nuclear a R corto y pérdida de enlace a R largo
    #           (forma tipo Morse, con su mínimo en la distancia de equilibrio)
    #   - c_XX: el solapamiento de orbitales -> la correlación se desvanece al separar los átomos
    #   - los términos de un cuerpo (IZ, ZI) se mantienen como energías orbitales de referencia
    Re = 0.735
    c_II = -1.05 + 0.75 * (1 - np.exp(-1.9 * (R - Re)))**2
    c_1q = 0.39
    c_ZZ = -0.01 * np.exp(-2.0 * (R - Re))
    c_XX = 0.18 * np.exp(-1.6 * (R - Re))
    return [c_II, c_1q, c_1q, c_ZZ, c_XX]

distancias = np.linspace(0.30, 2.60, 120)
energias = []
for R in distancias:
    Hr = SparsePauliOp.from_list(list(zip(terms, coeficientes_H2(R)))).to_matrix()
    energias.append(np.linalg.eigvalsh(Hr)[0])
energias = np.array(energias)

R_eq = distancias[np.argmin(energias)]
print(f"Distancia de equilibrio encontrada : {R_eq:.3f} Angstrom   (valor experimental H2: 0.741 A)")
print(f"Energía de disociación (R -> inf)  : {energias[-1]:.4f} Hartree")
print(f"Energía mínima del modelo          : {energias.min():.4f} Hartree")
print("\nInterpretación: buscar el MÍNIMO EIGENVALUE en cada R, y luego el mínimo sobre R,")
print("es exactamente 'encontrar la geometría molecular estable'.")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(distancias, energias, lw=2, color="#1f77b4", label="E(R) = eigenvalue mínimo de H(R)")
ax.axvline(R_eq, ls="--", color="crimson", label=f"Equilibrio ≈ {R_eq:.3f} Å")
ax.plot(R_eq, energias.min(), "o", color="crimson", ms=8)
ax.annotate("Estado fundamental\n(geometría estable)",
            xy=(R_eq, energias.min()), xytext=(R_eq + 0.55, energias.min() + 0.35),
            arrowprops=dict(arrowstyle="->", color="crimson"), color="crimson")
ax.set_xlabel("Distancia interatómica R (Å)")
ax.set_ylabel("Energía (Hartree)")
ax.set_title("Curva de energía potencial de H₂ — el mínimo define el enlace")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


## ✅ Respuesta al Reto 2

**Pregunta:** *¿De qué manera la búsqueda del estado fundamental en un hamiltoniano mapeado a qubits difiere
funcionalmente de la minimización clásica por métodos Hartree-Fock?*

**Respuesta:**

Ambos métodos buscan lo mismo (el mínimo de energía), pero **minimizan sobre espacios distintos**:

| | **Hartree-Fock (clásico)** | **Estado fundamental en qubits (VQE / diagonalización)** |
|---|---|---|
| **Qué se minimiza** | Energía sobre el conjunto de **determinantes de Slater únicos** (un solo producto antisimetrizado de orbitales). | Energía sobre **todo el espacio de Hilbert**: cualquier superposición de configuraciones. |
| **Modelo físico** | **Campo medio**: cada electrón siente el *promedio* de los demás. | Interacción **explícita**, electrón a electrón, instante a instante. |
| **Lo que se pierde/gana** | Ignora la **correlación electrónica** → siempre da una energía **más alta** que la real. La diferencia `E_exacta − E_HF` se llama *energía de correlación*. | Captura la correlación: es el término `XX` que mezcla `|00⟩` y `|11⟩` en el estado fundamental. |
| **Procedimiento** | Iteración autoconsistente (SCF): resolver las ecuaciones de Fock hasta que los orbitales dejan de cambiar. | **Variacional/espectral**: preparar `|ψ(θ)⟩` con un circuito, medir `⟨ψ|H|ψ⟩` y optimizar θ (VQE); o diagonalizar `H` cuando es pequeño. |
| **Escalado** | Barato: ≈ N⁴ (y por eso es el punto de partida de casi todo). | El coste clásico exacto (Full-CI) crece **exponencialmente**; en hardware cuántico el estado vive "gratis" en los qubits y el coste se traslada al **número de mediciones** y al ruido. |
| **Naturaleza del resultado** | Determinista, siempre converge al mismo mínimo de campo medio. | Estadístico: cada `⟨Pᵢ⟩` se estima con muestreo (shots) → hay incertidumbre estadística + ruido de hardware. |

**Diferencia funcional clave en una frase:** Hartree-Fock **restringe la forma de la función de onda** para que el
problema sea tratable clásicamente y paga el precio de perder la correlación; el enfoque cuántico **no restringe
la función de onda**, la representa directamente en los qubits, y paga el precio en **mediciones y ruido**.

**Sobre la distancia de enlace:** al variar R cambian las integrales moleculares y, por tanto, **los coeficientes**
de cada término de Pauli (la repulsión nuclear `~1/R` entra en `II`; el solapamiento que alimenta `XX` decae con R).
Encontrar el eigenvalue mínimo para cada R y quedarse con el mínimo global **es** hallar la geometría estable:
el valor propio mínimo es la energía del estado que la naturaleza escoge, y la geometría que lo produce es la
longitud de enlace real.


---
# RETO 3 — Mitigación de ruido y realidad NISQ (ZNE / PEA)

## 🧠 Entiende esto primero
- **NISQ** (*Noisy Intermediate-Scale Quantum*): la etapa actual. Cientos de qubits, **sin corrección de errores**.
- **Decoherencia:** el qubit "olvida" su estado al interactuar con el entorno (tiempos T₁ y T₂, de microsegundos).
- **Error de compuerta:** cada compuerta de 2 qubits (CNOT) falla hoy con probabilidad ~10⁻³–10⁻².
- **ZNE (Zero Noise Extrapolation):** no se puede *quitar* ruido, pero sí **añadir más a propósito** (factores
  λ = 1, 2, 3 …), ver cómo se degrada la energía, ajustar una curva y **extrapolar hacia λ = 0**.
  Cómo se amplifica el ruido: *unitary folding*, es decir, sustituir cada compuerta `U` por `U U† U` — que
  matemáticamente es lo mismo, pero físicamente ejecuta 3× más compuertas y por tanto 3× más ruido.
- **PEA (Probabilistic Error Amplification):** versión más fina: se caracteriza el ruido real del dispositivo y se
  **amplifica ese ruido concreto** de forma controlada antes de extrapolar. Más preciso, más caro en muestreo.
- **Mitigación ≠ corrección.** ZNE/PEA **reducen el sesgo**, no eliminan el error; la corrección de errores real
  (códigos de superficie) necesita muchos más qubits de los que hay hoy.


In [ ]:
# --- Simulación local de un experimento ZNE (sin hardware ni token) ---
E_exacta = float(np.linalg.eigvalsh(hamiltonian_simbolico.to_matrix())[0])

def energia_medida(lmbda, rng=None):
    # Modelo de ruido: el error crece ~linealmente con el factor de amplificación lambda.
    #   sesgo_residual = parte NO lineal del error (la que una ZNE lineal no puede eliminar)
    #   pendiente      = error que sí crece proporcional al número de compuertas ejecutadas
    sesgo_residual = 0.2105
    pendiente = 0.25
    ruido_estadistico = 0.0 if rng is None else rng.normal(0, 0.004)
    return E_exacta + sesgo_residual + pendiente * lmbda + ruido_estadistico

rng = np.random.default_rng(11)
factores = np.array([1.0, 2.0, 3.0])
medidas = np.array([energia_medida(l) for l in factores])

print(f"Energía exacta (referencia, sin ruido): {E_exacta:.4f} Ha\n")
print(f"{'Factor de ruido λ':<20}{'Energía medida (Ha)':>22}{'Error vs exacta':>18}")
print("-" * 62)
for l, e in zip(factores, medidas):
    print(f"{l:<20.1f}{e:>22.4f}{e - E_exacta:>18.4f}")
print("\nSe ve el patrón NISQ: a más ruido, la energía medida SUBE (se aleja del mínimo real).")


In [ ]:
# Extrapolación lineal a ruido cero: ajustamos E(λ) = a·λ + b  y evaluamos en λ = 0
a, b = np.polyfit(factores, medidas, 1)
E_mitigada = b          # el valor en λ = 0 es justamente la ordenada al origen
E_sin_mitigar = medidas[0]

print(f"Ajuste lineal: E(λ) = {a:+.4f}·λ {b:+.4f}\n")
print(f"Energía SIN mitigar (λ=1) : {E_sin_mitigar:.4f} Ha   (error {abs(E_sin_mitigar-E_exacta):.4f} Ha)")
print(f"Energía MITIGADA con ZNE  : {E_mitigada:.4f} Ha   (error {abs(E_mitigada-E_exacta):.4f} Ha)")
print(f"Energía exacta            : {E_exacta:.4f} Ha")

mejora = (1 - abs(E_mitigada - E_exacta) / abs(E_sin_mitigar - E_exacta)) * 100
print(f"\nZNE elimina el {mejora:.1f}% del error... pero NO todo:")

precision_quimica = 0.0016   # 1 kcal/mol en Hartree
print(f"Precisión química requerida: {precision_quimica} Ha (1 kcal/mol)")
print(f"Error residual tras ZNE    : {abs(E_mitigada-E_exacta):.4f} Ha "
      f"= {abs(E_mitigada-E_exacta)/precision_quimica:.0f}x MAYOR que el umbral aceptable.")
print("\nConclusión NISQ: mitigar ayuda muchísimo, pero todavía no alcanza la precisión química.")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
lam = np.linspace(0, 3.4, 100)
ax.plot(lam, a * lam + b, "--", color="gray", label="Ajuste lineal (extrapolación)")
ax.plot(factores, medidas, "o", ms=9, color="#d62728", label="Mediciones con ruido amplificado")
ax.plot(0, E_mitigada, "*", ms=18, color="#2ca02c", label=f"ZNE (λ=0): {E_mitigada:.4f} Ha")
ax.axhline(E_exacta, color="#1f77b4", lw=2, label=f"Energía exacta: {E_exacta:.4f} Ha")
ax.axhspan(E_exacta - 0.0016, E_exacta + 0.0016, color="#1f77b4", alpha=0.25,
           label="Banda de precisión química (±1 kcal/mol)")
ax.set_xlabel("Factor de amplificación de ruido  λ")
ax.set_ylabel("Energía estimada (Hartree)")
ax.set_title("Zero Noise Extrapolation: medir peor a propósito para estimar lo perfecto")
ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()


## ✅ Respuesta al Reto 3

**Cómo funciona ZNE (conceptualmente):** el ruido no se puede apagar, pero sí **escalar hacia arriba de forma
controlada**. Se ejecuta el mismo circuito con factores de ruido λ = 1, 2, 3 (típicamente con *unitary folding*:
`U → U U† U`, que no cambia la matemática pero triplica las compuertas físicas y, con ellas, el error). Se mide la
energía en cada caso, se observa que `E(λ)` se degrada de forma regular, se ajusta un modelo (lineal, polinómico o
exponencial) y se **evalúa ese modelo en λ = 0**, el punto que el hardware nunca puede visitar. Ese valor
extrapolado es la energía mitigada.

**Por qué corregir post-ejecución es preferible a calibrar físicamente cada compuerta de 2 qubits:**

1. **Escalado del coste de calibración.** El número de pares acoplados crece con el tamaño del chip; caracterizar y
   recalibrar cada compuerta de 2 qubits es un proceso lento (minutos a horas por dispositivo) que consume tiempo
   de máquina que no produce resultados.
2. **La calibración se degrada sola.** Los parámetros derivan (*drift*) en cuestión de horas por fluctuaciones
   térmicas y TLS; una calibración perfecta a las 9:00 ya no lo es a las 13:00. ZNE se aplica **en el mismo
   experimento**, así que usa el ruido que había *en ese momento*.
3. **Hay ruido que la calibración no toca.** Decoherencia (T₁/T₂), crosstalk, errores de lectura y ruido
   correlacionado no son errores de *ajuste*: no desaparecen calibrando mejor.
4. **Es agnóstica del hardware y componible.** ZNE es software: funciona sobre cualquier backend, se combina con
   *readout mitigation*, *dynamical decoupling* o PEA, y se puede afinar después de tomar los datos.
5. **Rendimiento decreciente de la perfección física.** Bajar el error de compuerta de 10⁻³ a 10⁻⁴ exige años de
   ingeniería; ZNE reduce el sesgo hoy, a cambio de **más shots** (coste de muestreo), que es un recurso
   escalable y comprable.

**Pregunta del reto — ¿por qué el error acumulado degrada la precisión necesaria para fármacos y catalizadores?**

Porque en química **lo que importa son diferencias de energía, no energías absolutas**:

- La **precisión química** es ~1 kcal/mol = **0.0016 Ha**. Con ese umbral se decide si una reacción ocurre o no,
  qué confórmero de un fármaco se une a la proteína, o qué barrera tiene un catalizador.
- Una barrera de activación se calcula como `E_estado_transición − E_reactivos`: dos números grandes (decenas de
  Hartree) cuya **diferencia** es diminuta. Un error de 0.2 Ha, como el del cálculo de arriba, es ~125 veces mayor
  que el umbral: la resta pierde todo su significado.
- La **velocidad de reacción depende exponencialmente** de esa barrera (Arrhenius, `k ∝ e^{−ΔE/kT}`): un error de
  1 kcal/mol ya cambia la constante de velocidad por un factor ≈ 5 a temperatura ambiente; un error de 0.2 Ha
  (≈ 125 kcal/mol) la cambia en decenas de órdenes de magnitud.
- Además el error **se acumula**: cada término de Pauli se mide por separado, `⟨H⟩ = Σ cᵢ⟨Pᵢ⟩`, y los sesgos de
  cientos o miles de términos se suman; y el circuito ansatz tiene profundidad creciente, así que a más compuertas,
  más decoherencia.

**Conclusión:** sin mitigación (y en última instancia sin corrección de errores), los resultados NISQ sirven para
demostrar el método, pero **no para tomar decisiones de diseño molecular**: predecirían la química equivocada.


---
# RETO 4 — Matriz de decisión: ¿cuándo migrar a la nube cuántica?

## 🧠 Entiende esto primero
- **Espacio de Hilbert:** el espacio de estados. Con `N` qubits tiene dimensión **2ᴺ**. Guardar el estado denso
  requiere 2ᴺ números complejos; guardar el **operador** denso requiere (2ᴺ)² = 4ᴺ.
- **Sistema fuertemente correlacionado:** aquel donde la aproximación de campo medio falla (metales de transición,
  clústeres **hierro-azufre** de la nitrogenasa, catalizadores, estados de capa abierta). Ahí DFT/HF dan errores
  cualitativos, no solo cuantitativos.
- **DFT (Density Functional Theory):** el caballo de batalla clásico, escala ≈ N³–N⁴ y funciona muy bien... hasta
  que hay correlación estática fuerte.
- **Umbral de ventaja cuántica:** el punto donde el mejor método clásico disponible deja de ser viable **o**
  deja de ser fiable, y el cuántico (con su coste de shots y mitigación) sale a cuenta.


In [ ]:
# Muro de escalabilidad: memoria para representar el HAMILTONIANO DENSO de N qubits
# Matriz de (2^N x 2^N) números complejos de 16 bytes cada uno.
BYTES_COMPLEJO = 16
TB = 2**40   # 1 TiB

print(f"{'Qubits':>7} | {'Dim (2^N)':>22} | {'Elementos (4^N)':>16} | {'RAM necesaria (TB)':>26}")
print("-" * 82)
for n in [10, 20, 30, 35, 40, 45, 50]:
    dim = 2**n
    elementos = dim * dim
    ram_tb = elementos * BYTES_COMPLEJO / TB
    print(f"{n:>7} | {dim:>22,} | {elementos:>16.3e} | {ram_tb:>26,.0f}")

print("\nReferencias del mundo real (RAM disponible):")
print("  Laptop potente (64 GB)      :          0.06 TB")
print("  Nodo HPC de gama alta       :         12    TB")
print("  Supercomputadora Frontier   :      9,200    TB  (~9.2 PB)")
print("\nCada qubit extra MULTIPLICA POR 4 la memoria del operador denso.")


In [ ]:
# ¿Dónde está la frontera práctica?
RAM_FRONTIER_TB = 9.2e3     # ~9.2 PB expresados en TB
RAM_HPC_TB = 12.0

def ram_tb(n):
    n = int(n)
    return (2**n) * (2**n) * BYTES_COMPLEJO / TB   # enteros exactos de Python, sin overflow

print("Máximo nº de qubits que cabe en cada máquina, con MATRIZ DENSA (peor caso):\n")
for etiqueta, limite in [("Laptop 64 GB", 64/1024), ("Nodo HPC 12 TB", RAM_HPC_TB),
                         ("Frontier ~9.2 PB", RAM_FRONTIER_TB)]:
    n = 1
    while ram_tb(n + 1) <= limite:
        n += 1
    print(f"{etiqueta:<20} -> máximo ~{n} qubits con matriz densa")

print("\nNota importante: con técnicas inteligentes (vector de estado en vez de matriz densa,")
print("tensor networks, DMRG, muestreo estocástico) el límite clásico real sube bastante")
print("(~45-50 qubits en estado-vector). El muro exponencial sigue estando ahí: solo se retrasa.")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ns = np.arange(4, 56)
ax.semilogy(ns, [ram_tb(int(n)) for n in ns], lw=2, color="#d62728", label="RAM matriz densa (TB)")
ax.axhline(64/1024, ls="--", color="#2ca02c", label="Laptop 64 GB (0.06 TB)")
ax.axhline(RAM_HPC_TB, ls="--", color="#ff7f0e", label="Nodo HPC 12 TB")
ax.axhline(RAM_FRONTIER_TB, ls="--", color="#1f77b4", label="Frontier ~9.2 PB (9,200 TB)")
ax.axvspan(45, 56, color="red", alpha=0.10)
ax.text(48, 1e8, "imposible\nclásicamente", ha="center", color="darkred", fontsize=9)
ax.set_xlabel("Número de qubits / espín-orbitales (N)")
ax.set_ylabel("RAM requerida (TB, escala log)")
ax.set_title("El muro exponencial: 4ᴺ crece más rápido que cualquier presupuesto")
ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# Matriz de decisión: ¿clásico o cuántico?
casos = [
    # (caso, orbitales activos, ¿correlación fuerte?, ¿precisión química exigida?)
    ("H2, H2O, moléculas orgánicas pequeñas",      "< 20", "No",  "Clásico (DFT/CCSD(T)) — barato y fiable"),
    ("Screening masivo de fármacos (docking)",     "miles","No",  "Clásico (DFT/MM) — el volumen mata al cuántico"),
    ("Catalizador de metal de transición (1 centro)","30-50","Sí", "Frontera — evaluar caso a caso"),
    ("Clúster FeMo-co de la nitrogenasa",          "> 100","Sí",  "Candidato cuántico — CASSCF clásico ya no llega"),
    ("Superconductores / materiales correlacionados","> 100","Sí", "Candidato cuántico"),
]

print(f"{'Caso':<46} | {'Orbitales':<9} | {'Corr. fuerte':<12} | Recomendación")
print("-" * 128)
for caso, orb, corr, rec in casos:
    print(f"{caso:<46} | {orb:<9} | {corr:<12} | {rec}")

print("\n\nRED FLAGS -> el problema debe QUEDARSE en clásico (DFT):")
for f in [
    "El sistema está bien descrito por un solo determinante (campo medio funciona).",
    "DFT ya reproduce el experimento dentro de 1 kcal/mol.",
    "El espacio activo cabe en < 30-40 orbitales -> CASSCF/DMRG clásico lo resuelve exacto.",
    "Se necesitan MILES de evaluaciones (screening, dinámica molecular): el coste de shots lo hace inviable.",
    "El error de hardware tras mitigación supera el efecto químico que se quiere medir.",
    "El cuello de botella es el tamaño del sistema (solvente, proteína completa), no la correlación.",
]:
    print("  x", f)

print("\nGREEN FLAGS -> justifica migrar a hardware cuántico:")
for f in [
    "Correlación estática fuerte: varios determinantes con peso comparable (multirreferencial).",
    "Espacio activo necesario > 50 orbitales: Full-CI clásico es imposible.",
    "DFT da resultados que dependen fuertemente del funcional elegido (señal de que no es fiable).",
    "Pocas evaluaciones de alto valor (una barrera de reacción clave vale millones en I+D).",
    "El valor económico de acertar supera con creces el coste de tiempo de QPU + mitigación.",
]:
    print("  v", f)


## ✅ Respuesta al Reto 4

**Pregunta:** *¿Qué criterio técnico y económico define el punto de inflexión exacto (quantum advantage threshold)
en el cual un equipo de I+D de materiales debe migrar su modelo molecular a un procesador cuántico?*

**Respuesta — el umbral tiene dos condiciones, y deben cumplirse las DOS a la vez:**

### A) Criterio técnico — "el método clásico ya no puede, o ya no es fiable"
1. **Intratabilidad, no solo lentitud.** El espacio activo necesario supera lo que el mejor método clásico exacto
   puede abordar. En la práctica: **más de ~50 espín-orbitales correlacionados**, donde Full-CI (dimensión ≈ 4ᴺ)
   excede la RAM del planeta (la tabla de arriba: 45 qubits ⇒ ~1.8 × 10¹⁶ TB) y donde DMRG/tensor networks también
   fallan porque el entrelazamiento no es de baja dimensión.
2. **Correlación estática fuerte (el criterio de calidad, más importante que el de tamaño).** Si la función de onda
   necesita **múltiples determinantes con peso comparable** — clústeres Fe-S de la nitrogenasa, centros de metales
   de transición, estados de capa abierta, materiales fuertemente correlacionados — DFT y CCSD(T) no fallan por
   unos decimales: fallan **cualitativamente**. Síntoma diagnóstico típico: el resultado cambia mucho según el
   funcional DFT elegido, o el diagnóstico T1 de coupled-cluster es alto.
3. **Que el cuántico realmente entregue.** El error **tras mitigación** (ZNE/PEA) debe ser menor que el efecto
   químico que se quiere resolver, idealmente dentro de la precisión química (0.0016 Ha ≈ 1 kcal/mol). Como muestra
   el Reto 3, **hoy esta condición todavía no se cumple de forma rutinaria**, y es la que realmente frena la
   migración.

### B) Criterio económico — "vale más de lo que cuesta"
1. **Coste total cuántico** = tiempo de QPU × número de circuitos × shots por circuito × sobrecoste de mitigación.
   ZNE multiplica el muestreo por ~3–5×; PEA puede multiplicarlo **exponencialmente** con la profundidad del
   circuito. Ese sobrecoste de muestreo, no el número de qubits, es hoy el verdadero factor limitante del coste.
2. **Pocas evaluaciones de alto valor, no muchas baratas.** El cuántico solo sale a cuenta cuando se necesitan
   **decenas** de cálculos decisivos (una barrera de reacción, un mecanismo catalítico), no miles (screening de
   fármacos o dinámica molecular): el coste por punto es demasiado alto para el volumen.
3. **Valor de la decisión.** Un catalizador mejorado para la síntesis de amoníaco o un electrolito de batería valen
   cientos de millones; ahí un cálculo caro pero correcto se amortiza. Un cálculo rutinario de una molécula
   orgánica pequeña, no.

### El punto de inflexión, formulado de forma operativa
> **Se debe migrar cuando el problema es simultáneamente (a) fuertemente correlacionado y demasiado grande para
> el mejor método clásico exacto (> ~50 orbitales activos), (b) resoluble en hardware actual con error post-mitigación
> menor que el efecto químico buscado, y (c) de valor económico suficiente para absorber el sobrecoste de muestreo
> y mitigación.**
>
> Si falla **cualquiera** de las tres, la respuesta correcta hoy sigue siendo **quedarse en DFT / CCSD(T) / DMRG**.

**Matiz honesto (y esperado en la evaluación):** el hito de 2023 con el procesador de 127 qubits demostró
*utilidad cuántica* — valores de expectación precisos en circuitos fuera del alcance de la simulación clásica por
fuerza bruta — **en una tarea concreta y bien definida**, no superioridad general. Para química molecular de
producción, el umbral aún no se ha cruzado de forma rutinaria: la ruta realista es **híbrida** (clásico para lo
que el clásico hace bien, cuántico para el espacio activo fuertemente correlacionado).


---
# Cierre: las 5 ideas que tienes que poder decir de memoria

1. **El Hamiltoniano es una matriz de energía.** Su **eigenvalue mínimo** = estado fundamental = molécula estable.
2. **Se escribe como suma ponderada de cadenas de Pauli** porque las Paulis son base completa del espacio de
   operadores, y porque es el único formato que el hardware cuántico sabe **medir** directamente.
3. **Los términos `Z` son ocupación (diagonal, estática); los `X`/`Y` son saltos y correlación (fuera de la
   diagonal).** La correlación es justo lo que Hartree-Fock no ve.
4. **La era NISQ impone un techo de precisión.** ZNE amplifica el ruido a propósito y extrapola a λ = 0; mitiga,
   pero no alcanza aún la precisión química de 1 kcal/mol de forma rutinaria.
5. **El muro clásico es 4ᴺ.** Migrar a cuántico se justifica solo cuando hay correlación fuerte + tamaño
   intratable + valor económico que absorba el coste de shots y mitigación.

### Preguntas de reflexión del cierre del tema (por si las piden)
- **Aplicación industrial real:** diseño de catalizadores para la fijación de nitrógeno (sustituir Haber-Bosch,
  que consume ~1-2% de la energía mundial), electrolitos y cátodos para baterías de estado sólido, y materiales
  para captura de carbono. Todos son problemas con correlación electrónica fuerte donde DFT es poco fiable.
- **Limitaciones actuales y cómo superarlas:** ruido y decoherencia (→ mitigación ZNE/PEA a corto plazo, corrección
  de errores con códigos de superficie a medio plazo), coste de muestreo (→ mejores agrupaciones de términos de
  Pauli y algoritmos con menos mediciones), y número de qubits lógicos (→ mejor conectividad y qubits de mayor
  fidelidad).
